In [1]:
# =============================================================
# Cargar datos, preprocesamiento, modelos y top3 del Sprint 3
# =============================================================

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# IMPORTANTE: Pipeline correcto para sampling Y para preprocesamiento
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# =============================================================
# 1. Cargar conjunto de datos
# =============================================================
df = pd.read_csv("../data/processed/04_default_credit_features.csv")
df = df.drop(columns=["ID"])

TARGET = "default payment next month"
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Columnas
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# =============================================================
# 2. Preprocesamiento (usando imblearn.Pipeline)
# =============================================================
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preproc = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

# =============================================================
# 3. Modelos optimizados (Sprint 3)
# =============================================================
modelos = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "DecisionTree": DecisionTreeClassifier(max_depth=5),
    "RandomForest": RandomForestClassifier(n_estimators=20, n_jobs=-1),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "GradientBoosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=50, n_jobs=-1)
}

# =============================================================
# 4. Balanceo optimizado
# =============================================================
estrategias = {
    "NoBalance": None,
    "Over": RandomOverSampler(random_state=42)
}

# =============================================================
# 5. Validación
# =============================================================
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
metricas = ["accuracy", "precision", "recall", "f1", "roc_auc"]
resultados = {}

# =============================================================
# 6. Entrenamiento (Sprint 3)
# =============================================================
for nombre_modelo, modelo in modelos.items():
    for nombre_estrategia, sampler in estrategias.items():

        steps = [("preprocessing", preproc)]

        if sampler is not None:
            steps.append(("sampling", sampler))

        steps.append(("clf", modelo))

        # Pipeline CORRECTO (imblearn)
        pipe = Pipeline(steps)

        puntuaciones = cross_validate(
            pipe, X, y, cv=cv, scoring=metricas, n_jobs=-1
        )

        resultados[f"{nombre_modelo}_{nombre_estrategia}"] = {
            m: puntuaciones[f"test_{m}"].mean() for m in metricas
        }

# =============================================================
# 7. Ranking y Top 3
# =============================================================
tabla = pd.DataFrame(resultados).T
ranking = tabla.sort_values(by="f1", ascending=False)
top3 = ranking.head(3)

print("=== Top 3 modelos ===")
display(top3)

=== Top 3 modelos ===


,accuracy,precision,recall,f1,roc_auc
SVM_Over,0.773933,0.491030,0.597498,0.539021,0.765311
GradientBoosting_Over,0.762733,0.472481,0.622815,0.537323,0.780334
AdaBoost_Over,0.766300,0.477477,0.595690,0.530030,0.770839


In [2]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

print("=== Sprint 4: Hyperparameter Tuning (Optimizado) ===")

# =============================================================
# 1. Espacios de búsqueda reducidos
# =============================================================
param_spaces = {
    "SVM_Over": {
        "clf__C": [0.1, 1],
        "clf__kernel": ["linear", "rbf"],
        "clf__gamma": ["scale"]
    },
    "GradientBoosting_Over": {
        "clf__n_estimators": [50, 100],
        "clf__learning_rate": [0.05, 0.1],
        "clf__max_depth": [2, 3]
    },
    "AdaBoost_Over": {
        "clf__n_estimators": [50, 100],
        "clf__learning_rate": [0.05, 0.1]
    }
}

# =============================================================
# 2. CV estratificado rápido
# =============================================================
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# =============================================================
# 3. Grid + Random
# =============================================================
tuning_results = {}
os.makedirs("models/tuned", exist_ok=True)

for modelo_nombre in top3.index:
    print(f"\n--- Tuning para {modelo_nombre} ---")

    modelo_base = modelos[modelo_nombre.split("_")[0]]
    grid = param_spaces.get(modelo_nombre, None)

    if grid is None:
        print(f"No hay grid definido para {modelo_nombre}, se omite.")
        continue

    pipe = Pipeline([
        ("preprocessing", preproc),
        ("sampling", RandomOverSampler(random_state=42)),
        ("clf", modelo_base)
    ])

    # ---------------- GRID SEARCH ----------------
    grid_search = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1
    )
    grid_search.fit(X, y)

    # ---------------- RANDOM SEARCH ----------------
    random_search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=grid,
        n_iter=5,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        random_state=42
    )
    random_search.fit(X, y)

    # ---------------- GUARDAR RESULTADOS ----------------
    tuning_results[modelo_nombre] = {
        "grid_best_params": grid_search.best_params_,
        "grid_best_score": grid_search.best_score_,
        "random_best_params": random_search.best_params_,
        "random_best_score": random_search.best_score_
    }

    # ---------------- IMPRIMIR ----------------
    print("GridSearch mejor:", grid_search.best_params_, "Score:", grid_search.best_score_)
    print("RandomSearch mejor:", random_search.best_params_, "Score:", random_search.best_score_)

# Resumen general
print("\n=== Resumen Grid + Random ===")
print(pd.DataFrame(tuning_results).T)


=== Sprint 4: Hyperparameter Tuning (Optimizado) ===

--- Tuning para SVM_Over ---


C:\Users\User\miniforge3\envs\dp261-g3_Project\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=5. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


GridSearch mejor: {'clf__C': 1, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'} Score: 0.5403874989133235
RandomSearch mejor: {'clf__kernel': 'rbf', 'clf__gamma': 'scale', 'clf__C': 1} Score: 0.5403874989133235

--- Tuning para GradientBoosting_Over ---
GridSearch mejor: {'clf__learning_rate': 0.1, 'clf__max_depth': 2, 'clf__n_estimators': 50} Score: 0.5377517126614797
RandomSearch mejor: {'clf__n_estimators': 100, 'clf__max_depth': 2, 'clf__learning_rate': 0.1} Score: 0.5375760615035567

--- Tuning para AdaBoost_Over ---


C:\Users\User\miniforge3\envs\dp261-g3_Project\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=5. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


GridSearch mejor: {'clf__learning_rate': 0.1, 'clf__n_estimators': 100} Score: 0.5230306389495217
RandomSearch mejor: {'clf__n_estimators': 100, 'clf__learning_rate': 0.1} Score: 0.5230306389495217

=== Resumen Grid + Random ===
                                                        grid_best_params  \
SVM_Over               {'clf__C': 1, 'clf__gamma': 'scale', 'clf__ker...   
GradientBoosting_Over  {'clf__learning_rate': 0.1, 'clf__max_depth': ...   
AdaBoost_Over          {'clf__learning_rate': 0.1, 'clf__n_estimators...   

                      grid_best_score  \
SVM_Over                     0.540387   
GradientBoosting_Over        0.537752   
AdaBoost_Over                0.523031   

                                                      random_best_params  \
SVM_Over               {'clf__kernel': 'rbf', 'clf__gamma': 'scale', ...   
GradientBoosting_Over  {'clf__n_estimators': 100, 'clf__max_depth': 2...   
AdaBoost_Over          {'clf__n_estimators': 100, 'clf__learning_rate... 

In [4]:
pip install optuna

  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
Using cached alembic-1.18.4-py3-none-any.whl (263 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 60.7 MB/s  0:00:00

   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   ------------- -------------------------- 2/6 [colorlog]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   ------------------

In [5]:
import optuna
from sklearn.metrics import f1_score

def objective(trial):
    params = {}
    for key, values in grid.items():
        if isinstance(values[0], float):
            params[key] = trial.suggest_float(key, min(values), max(values))
        elif isinstance(values[0], int):
            params[key] = trial.suggest_int(key, min(values), max(values))
        else:
            params[key] = trial.suggest_categorical(key, values)

    pipe_opt = Pipeline([
        ("preprocessing", preproc),
        ("sampling", RandomOverSampler(random_state=42)),
        ("clf", modelo_base.set_params(**{
            k.replace("clf__", ""): v for k, v in params.items()
        }))
    ])

    pipe_opt.fit(X, y)
    y_pred = pipe_opt.predict(X)
    return f1_score(y, y_pred, average="macro")

# Ejecutar Optuna (ejemplo para un modelo)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

print("\n=== Optuna Resultados ===")
print("Mejores parámetros:", study.best_params)
print("Mejor score:", study.best_value)


C:\Users\User\miniforge3\envs\dp261-g3_Project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-06 23:22:25,646] A new study created in memory with name: no-name-3014455c-75c2-48cf-b5a1-dd1a02db8c5c
[I 2026-05-06 23:22:32,249] Trial 0 finished with value: 0.6840138021902513 and parameters: {'clf__n_estimators': 75, 'clf__learning_rate': 0.062017023215441926}. Best is trial 0 with value: 0.6840138021902513.
[I 2026-05-06 23:22:40,074] Trial 1 finished with value: 0.6904642554021525 and parameters: {'clf__n_estimators': 90, 'clf__learning_rate': 0.0818953594173435}. Best is trial 1 with value: 0.6904642554021525.
[I 2026-05-06 23:22:48,184] Trial 2 finished with value: 0.6904642554021525 and parameters: {'clf__n_estimators': 91, 'clf__learning_rate': 0.08153046753668353}. Best is trial 1 with value: 0.6904


=== Optuna Resultados ===
Mejores parámetros: {'clf__n_estimators': 90, 'clf__learning_rate': 0.0818953594173435}
Mejor score: 0.6904642554021525


In [6]:
# 4. Comparación before vs after
# =============================================================
before = top3.loc[modelo_nombre]["f1"]
after = max(grid_search.best_score_, random_search.best_score_, study.best_value)

# =============================================================
# 5. Guardar modelo tuneado
# =============================================================
best_params = study.best_params
best_model = modelo_base.set_params(**{k.replace("clf__", ""): v for k, v in best_params.items()})

final_pipe = Pipeline([
        ("preprocessing", preproc),
        ("sampling", RandomOverSampler(random_state=42)),
        ("clf", best_model)
    ])

final_pipe.fit(X, y)
joblib.dump(final_pipe, f"models/tuned/{modelo_nombre}_tuned.pkl")

# =============================================================
# 6. Documentar resultados
# =============================================================
tuning_results[modelo_nombre] = {
        "before_f1": before,
        "after_f1": after,
        "best_params_grid": grid_search.best_params_,
        "best_params_random": random_search.best_params_,
        "best_params_optuna": study.best_params
    }

# =============================================================
# Tabla final de resultados
# =============================================================
tuning_df = pd.DataFrame(tuning_results).T
print("\n=== Resultados Finales Sprint 4 (Optimizado) ===")
display(tuning_df)



=== Resultados Finales Sprint 4 (Optimizado) ===


,grid_best_params,grid_best_score,random_best_params,random_best_score,before_f1,after_f1,best_params_grid,best_params_random,best_params_optuna
SVM_Over,"{'clf__C': 1, 'clf__gamma': 'scale', 'clf__ker...",0.540387,"{'clf__kernel': 'rbf', 'clf__gamma': 'scale', ...",0.540387,NaN,NaN,NaN,NaN,NaN
GradientBoosting_Over,"{'clf__learning_rate': 0.1, 'clf__max_depth': ...",0.537752,"{'clf__n_estimators': 100, 'clf__max_depth': 2...",0.537576,NaN,NaN,NaN,NaN,NaN
AdaBoost_Over,NaN,NaN,NaN,NaN,0.53003,0.690464,"{'clf__learning_rate': 0.1, 'clf__n_estimators...","{'clf__n_estimators': 100, 'clf__learning_rate...","{'clf__n_estimators': 90, 'clf__learning_rate'..."


In [8]:
import optuna
import joblib
import pandas as pd

# =============================================================
# Continuación Sprint 4 - Optuna + before vs after
# =============================================================

for modelo_nombre in top3.index:

    print(f"\n=========== OPTUNA PARA {modelo_nombre} ===========")

    # =========================================================
    # Modelo base
    # =========================================================
    modelo_base = modelos[modelo_nombre.split("_")[0]]

    # =========================================================
    # Grid correspondiente
    # =========================================================
    grid = param_spaces[modelo_nombre]

    # =========================================================
    # FUNCIÓN OBJETIVO
    # =========================================================
    def objective(trial):

        params = {}

        for key, values in grid.items():

            # Parámetros float
            if isinstance(values[0], float):

                params[key] = trial.suggest_float(
                    key,
                    min(values),
                    max(values)
                )

            # Parámetros enteros
            elif isinstance(values[0], int):

                params[key] = trial.suggest_int(
                    key,
                    min(values),
                    max(values)
                )

            # Parámetros categóricos
            else:

                params[key] = trial.suggest_categorical(
                    key,
                    values
                )

        # =====================================================
        # Pipeline con preprocessing + oversampling + modelo
        # =====================================================
        pipe_opt = Pipeline([
            ("preprocessing", preproc),
            ("sampling", RandomOverSampler(random_state=42)),
            ("clf", modelo_base.set_params(**{
                k.replace("clf__", ""): v
                for k, v in params.items()
            }))
        ])

        # =====================================================
        # Cross Validation
        # =====================================================
        scores = cross_validate(
            pipe_opt,
            X,
            y,
            cv=cv,
            scoring="f1",
            n_jobs=-1
        )

        return scores["test_score"].mean()

    # =========================================================
    # EJECUTAR OPTUNA
    # =========================================================
    study = optuna.create_study(direction="maximize")

    study.optimize(
        objective,
        n_trials=5
    )

    print("Best Optuna Params:", study.best_params)
    print("Best Optuna Score:", study.best_value)

    # =========================================================
    # BEFORE VS AFTER
    # =========================================================
    before = top3.loc[modelo_nombre]["f1"]

    # Corrección del KeyError
    grid_score = tuning_results.get(
        modelo_nombre, {}
    ).get("grid_best_score", 0)

    random_score = tuning_results.get(
        modelo_nombre, {}
    ).get("random_best_score", 0)

    after = max(
        grid_score,
        random_score,
        study.best_value
    )

    # =========================================================
    # GUARDAR MODELO FINAL
    # =========================================================
    best_model = modelo_base.set_params(**{
        k.replace("clf__", ""): v
        for k, v in study.best_params.items()
    })

    final_pipe = Pipeline([
        ("preprocessing", preproc),
        ("sampling", RandomOverSampler(random_state=42)),
        ("clf", best_model)
    ])

    final_pipe.fit(X, y)

    joblib.dump(
        final_pipe,
        f"models/tuned/{modelo_nombre}_tuned.pkl"
    )

    # =========================================================
    # ACTUALIZAR RESULTADOS
    # =========================================================
    if modelo_nombre not in tuning_results:
        tuning_results[modelo_nombre] = {}

    tuning_results[modelo_nombre]["before_f1"] = before
    tuning_results[modelo_nombre]["after_f1"] = after
    tuning_results[modelo_nombre]["best_params_optuna"] = study.best_params

# =============================================================
# TABLA FINAL
# =============================================================
tuning_df = pd.DataFrame(tuning_results).T

print("\n=== RESULTADOS FINALES COMPLETOS ===")

display(tuning_df)

[I 2026-05-07 00:19:08,640] A new study created in memory with name: no-name-5191ee73-4ecb-4918-9009-71ef479a693c



=========== OPTUNA PARA SVM_Over ===========


[I 2026-05-07 00:21:26,515] Trial 0 finished with value: 0.5421608467891778 and parameters: {'clf__C': 0.6737768514474578, 'clf__kernel': 'rbf', 'clf__gamma': 'scale'}. Best is trial 0 with value: 0.5421608467891778.
[I 2026-05-07 00:23:22,421] Trial 1 finished with value: 0.5154144660576723 and parameters: {'clf__C': 0.23762179088245206, 'clf__kernel': 'linear', 'clf__gamma': 'scale'}. Best is trial 0 with value: 0.5421608467891778.
[I 2026-05-07 00:25:34,429] Trial 2 finished with value: 0.5426714924890482 and parameters: {'clf__C': 0.5042172766465844, 'clf__kernel': 'rbf', 'clf__gamma': 'scale'}. Best is trial 2 with value: 0.5426714924890482.
[I 2026-05-07 00:27:49,045] Trial 3 finished with value: 0.5420982185549893 and parameters: {'clf__C': 0.17891089994693465, 'clf__kernel': 'rbf', 'clf__gamma': 'scale'}. Best is trial 2 with value: 0.5426714924890482.
[I 2026-05-07 00:30:02,724] Trial 4 finished with value: 0.5426931459209208 and parameters: {'clf__C': 0.5154495620203996, 'clf

Best Optuna Params: {'clf__C': 0.5154495620203996, 'clf__kernel': 'rbf', 'clf__gamma': 'scale'}
Best Optuna Score: 0.5426931459209208


[I 2026-05-07 00:40:16,414] A new study created in memory with name: no-name-c308463c-88e0-45fb-96cd-0ac974fdaef7


...

[I 2026-05-07 00:41:20,138] A new study created in memory with name: no-name-204f30fb-b343-4a95-b7b9-62aabba3095f



=========== OPTUNA PARA AdaBoost_Over ===========


[I 2026-05-07 00:41:25,496] Trial 0 finished with value: 0.5218584952043951 and parameters: {'clf__n_estimators': 81, 'clf__learning_rate': 0.09933173865473287}. Best is trial 0 with value: 0.5218584952043951.
[I 2026-05-07 00:41:29,839] Trial 1 finished with value: 0.5200593473955645 and parameters: {'clf__n_estimators': 100, 'clf__learning_rate': 0.07201755457261887}. Best is trial 0 with value: 0.5218584952043951.
[I 2026-05-07 00:41:34,041] Trial 2 finished with value: 0.509760726553978 and parameters: {'clf__n_estimators': 98, 'clf__learning_rate': 0.05772802140112399}. Best is trial 0 with value: 0.5218584952043951.
[I 2026-05-07 00:41:37,239] Trial 3 finished with value: 0.509760726553978 and parameters: {'clf__n_estimators': 73, 'clf__learning_rate': 0.07063311494012714}. Best is trial 0 with value: 0.5218584952043951.
[I 2026-05-07 00:41:41,025] Trial 4 finished with value: 0.5200593473955645 and parameters: {'clf__n_estimators': 86, 'clf__learning_rate': 0.08757501981763172}.

Best Optuna Params: {'clf__n_estimators': 81, 'clf__learning_rate': 0.09933173865473287}
Best Optuna Score: 0.5218584952043951

=== RESULTADOS FINALES COMPLETOS ===


,grid_best_params,grid_best_score,random_best_params,random_best_score,before_f1,after_f1,best_params_optuna,best_params_grid,best_params_random
SVM_Over,"{'clf__C': 1, 'clf__gamma': 'scale', 'clf__ker...",0.540387,"{'clf__kernel': 'rbf', 'clf__gamma': 'scale', ...",0.540387,0.539021,0.542693,"{'clf__C': 0.5154495620203996, 'clf__kernel': ...",NaN,NaN
GradientBoosting_Over,"{'clf__learning_rate': 0.1, 'clf__max_depth': ...",0.537752,"{'clf__n_estimators': 100, 'clf__max_depth': 2...",0.537576,0.537323,0.537759,"{'clf__n_estimators': 69, 'clf__learning_rate'...",NaN,NaN
AdaBoost_Over,NaN,NaN,NaN,NaN,0.53003,0.521858,"{'clf__n_estimators': 81, 'clf__learning_rate'...","{'clf__learning_rate': 0.1, 'clf__n_estimators...","{'clf__n_estimators': 100, 'clf__learning_rate..."
